# 01 - Nettoyage des données brutes

**Objectif** : Charger les deux fichiers bruts ILOSTAT `data/raw/`, renommer les colonnes, appliquer les filtres de qualité, et exporter des fichiers propres dans `data/clean/`.

**Fichiers en entrée** :
- `data/raw/EMP_TEMP_SEX_EDU_NB_A.csv` — qui est l'emploi par sexe et éducation (annuel)
- `data/raw/UNE_TUNE_SEX_EDU_NB_A.csv` — qui est le chômage par sexe et éducation (annuel)

**Fichiers en sortie** :
- `data/clean/emploi_clean.csv`
- `data/clean/chomage_clean.csv`

## 0. Import

In [ ]:
import pandas as pd
import os
from pathlib import Path

# Chemin du projet et des données
PROJECT_ROOT = Path(r'C:\Users\HP\Documents\education-employment-analysis')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
CLEAN_DIR = PROJECT_ROOT / 'data' / 'clean'
FIGURES_DIR = PROJECT_ROOT / 'figures'

# Création des dossiers de sortie (S'IL N'EXISTENT PAS DEJA)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Prêt.')

Prêt.


## 1. Chargement des fichiers bruts

In [6]:
# Chargement sans aucune modification
emp_raw = pd.read_csv(RAW_DIR / 'EMP_TEMP_SEX_EDU_NB_A.csv')
une_raw = pd.read_csv(RAW_DIR / 'UNE_TUNE_SEX_EDU_NB_A.csv')

print('=== EMPLOI brut ===')
print(f'Dimensions : {emp_raw.shape}')
print(f'Colonnes   : {emp_raw.columns.tolist()}')
print()
print('=== CHÔMAGE brut ===')
print(f'Dimensions : {une_raw.shape}')
print(f'Colonnes   : {une_raw.columns.tolist()}')

=== EMPLOI brut ===
Dimensions : (92261, 11)
Colonnes   : ['ref_area', 'source', 'indicator', 'sex', 'classif1', 'time', 'obs_value', 'obs_status', 'note_classif', 'note_indicator', 'note_source']

=== CHÔMAGE brut ===
Dimensions : (88295, 11)
Colonnes   : ['ref_area', 'source', 'indicator', 'sex', 'classif1', 'time', 'obs_value', 'obs_status', 'note_classif', 'note_indicator', 'note_source']


In [7]:
# Aperçu des premières lignes (emploi)
emp_raw.head()

,ref_area,source,indicator,sex,classif1,time,obs_value,obs_status,note_classif,note_indicator,note_source
0,AFG,BA:15715,EMP_TEMP_SEX_EDU_NB,SEX_T,EDU_AGGREGATE_TOTAL,2021,7679.474,NaN,NaN,NaN,R1:3513_S3:8
1,AFG,BA:15715,EMP_TEMP_SEX_EDU_NB,SEX_T,EDU_AGGREGATE_LTB,2021,4916.311,NaN,NaN,NaN,R1:3513_S3:8
2,AFG,BA:15715,EMP_TEMP_SEX_EDU_NB,SEX_T,EDU_AGGREGATE_BAS,2021,1273.143,NaN,NaN,NaN,R1:3513_S3:8
3,AFG,BA:15715,EMP_TEMP_SEX_EDU_NB,SEX_T,EDU_AGGREGATE_INT,2021,940.560,NaN,NaN,NaN,R1:3513_S3:8
4,AFG,BA:15715,EMP_TEMP_SEX_EDU_NB,SEX_T,EDU_AGGREGATE_ADV,2021,448.800,NaN,NaN,NaN,R1:3513_S3:8


## 2. Compréhension des colonnes brutes

Les colonnes ILO ont des noms techniques qu'on va changer. Voici leur signification :

| Colonne brute | Signification | Nouveau nom |
|---|---|---|
| `ref_area` | Code ISO du pays (ex: FRA, DEU) | `country` |
| `source` | Source statistique nationale | *supprimée* |
| `indicator` | Code de l'indicateur ILO | `indicator` |
| `sex` | Sexe : SEX_T=total, SEX_M=hommes, SEX_F=femmes | `sex` |
| `classif1` | Code du niveau d'éducation ISCED | `education_code` |
| `time` | Année | `year` |
| `obs_value` | Valeur en milliers de personnes | `value` |
| `obs_status` | Qualité : NaN=ok, U=peu fiable, B=rupture de série | `obs_status` |
| `note_classif` | Notes sur la classification | *supprimée* |
| `note_indicator` | Notes sur l'indicateur | *supprimée* |
| `note_source` | Notes sur la source | *supprimée* |

## 3. Correspondance des codes d'éducation

Les données ILO utilisent la classification ISCED. On retient les codes **AGGREGATE** (meilleure couverture géographique) et on les regroupe en 3 niveaux lisibles.

In [8]:
# Codes ISCED retenus → niveau simplifié
# EDU_AGGREGATE_BAS = éducation primaire ou inférieure → LOW
# EDU_AGGREGATE_INT = éducation secondaire             → MEDIUM
# EDU_AGGREGATE_ADV = enseignement supérieur           → HIGH
CODES_EDUCATION = {
    'EDU_AGGREGATE_BAS': 'LOW',
    'EDU_AGGREGATE_INT': 'MEDIUM',
    'EDU_AGGREGATE_ADV': 'HIGH',
}

# Libellés lisibles pour les niveaux
LIBELLES_NIVEAU = {
    'LOW'   : 'Faible (primaire ou moins)',
    'MEDIUM': 'Intermédiaire (secondaire)',
    'HIGH'  : 'Élevé (supérieur)',
}

print('Codes éducation retenus :')
for code, niveau in CODES_EDUCATION.items():
    print(f'  {code:25s} → {niveau} ({LIBELLES_NIVEAU[niveau]})')

Codes éducation retenus :
  EDU_AGGREGATE_BAS         → LOW (Faible (primaire ou moins))
  EDU_AGGREGATE_INT         → MEDIUM (Intermédiaire (secondaire))
  EDU_AGGREGATE_ADV         → HIGH (Élevé (supérieur))


## 4. Fonction de nettoyage

In [9]:
def nettoyer_dataset(df_raw, nom_valeur):
    """
    Nettoie un dataset ILO brut (emploi ou chômage).

    Étapes :
    1. Filtre sur le sexe total (SEX_T), les codes éducation retenus,
       les valeurs non nulles, les données fiables (obs_status != U),
       et la période 2000-2023.
    2. Renomme les colonnes avec des noms lisibles.
    3. Ajoute la colonne 'education_level' (LOW / MEDIUM / HIGH).
    4. Supprime les colonnes techniques inutiles.

    Paramètres
    ----------
    df_raw    : DataFrame brut ILOSTAT
    nom_valeur: nom donné à la colonne obs_value (ex: 'nb_employes', 'nb_chomeurs')

    Retourne
    --------
    DataFrame nettoyé avec colonnes :
    country, year, sex, education_code, education_level, indicator, nom_valeur
    """

    # ── Étape 1 : filtrage ────────────────────────────────────────────
    df = df_raw[
        (df_raw['sex']       == 'SEX_T')                      # sexe : total
        & (df_raw['classif1'].isin(CODES_EDUCATION.keys()))   # codes éducation retenus
        & (df_raw['obs_value'].notna())                       # pas de valeur manquante
        & (df_raw['obs_status'] != 'U')                       # données fiables
        & (df_raw['time'] >= 2000)                            # période 2000–2023
        & (df_raw['time'] <= 2023)
    ].copy()

    # ── Étape 2 : renommage des colonnes ──────────────────────────────
    df.rename(columns={
        'ref_area' : 'country',
        'time'     : 'year',
        'sex'      : 'sex',
        'classif1' : 'education_code',
        'indicator': 'indicator',
        'obs_value': nom_valeur,
    }, inplace=True)

    # ── Étape 3 : ajout du niveau d'éducation simplifié ───────────────
    df['education_level'] = df['education_code'].map(CODES_EDUCATION)

    # ── Étape 4 : suppression des colonnes techniques inutiles ────────
    colonnes_a_garder = ['country', 'year', 'sex', 'education_code',
                         'education_level', 'indicator', nom_valeur]
    df = df[colonnes_a_garder].reset_index(drop=True)

    return df


print('Fonction nettoyer_dataset() définie.')

Fonction nettoyer_dataset() définie.


## 5. Application du nettoyage

In [10]:
# Nettoyage du dataset Emploi
emploi = nettoyer_dataset(emp_raw, 'nb_employes')

print('=== EMPLOI nettoyé ===')
print(f'Dimensions : {emploi.shape}')
print(f'Pays       : {emploi["country"].nunique()}')
print(f'Période    : {emploi["year"].min()} – {emploi["year"].max()}')
print(f'Colonnes   : {emploi.columns.tolist()}')
print()
emploi.head(6)

=== EMPLOI nettoyé ===
Dimensions : (6108, 7)
Pays       : 178
Période    : 2000 – 2023
Colonnes   : ['country', 'year', 'sex', 'education_code', 'education_level', 'indicator', 'nb_employes']



,country,year,sex,education_code,education_level,indicator,nb_employes
0,AFG,2021,SEX_T,EDU_AGGREGATE_BAS,LOW,EMP_TEMP_SEX_EDU_NB,1273.143
1,AFG,2021,SEX_T,EDU_AGGREGATE_INT,MEDIUM,EMP_TEMP_SEX_EDU_NB,940.560
2,AFG,2021,SEX_T,EDU_AGGREGATE_ADV,HIGH,EMP_TEMP_SEX_EDU_NB,448.800
3,AFG,2020,SEX_T,EDU_AGGREGATE_BAS,LOW,EMP_TEMP_SEX_EDU_NB,910.916
4,AFG,2020,SEX_T,EDU_AGGREGATE_INT,MEDIUM,EMP_TEMP_SEX_EDU_NB,693.897
5,AFG,2020,SEX_T,EDU_AGGREGATE_ADV,HIGH,EMP_TEMP_SEX_EDU_NB,386.350


In [11]:
# Nettoyage du dataset Chômage
chomage = nettoyer_dataset(une_raw, 'nb_chomeurs')

print('=== CHÔMAGE nettoyé ===')
print(f'Dimensions : {chomage.shape}')
print(f'Pays       : {chomage["country"].nunique()}')
print(f'Période    : {chomage["year"].min()} – {chomage["year"].max()}')
print(f'Colonnes   : {chomage.columns.tolist()}')
print()
chomage.head(6)

=== CHÔMAGE nettoyé ===
Dimensions : (5924, 7)
Pays       : 175
Période    : 2000 – 2023
Colonnes   : ['country', 'year', 'sex', 'education_code', 'education_level', 'indicator', 'nb_chomeurs']



,country,year,sex,education_code,education_level,indicator,nb_chomeurs
0,AFG,2021,SEX_T,EDU_AGGREGATE_BAS,LOW,UNE_TUNE_SEX_EDU_NB,81.747
1,AFG,2021,SEX_T,EDU_AGGREGATE_INT,MEDIUM,UNE_TUNE_SEX_EDU_NB,98.258
2,AFG,2021,SEX_T,EDU_AGGREGATE_ADV,HIGH,UNE_TUNE_SEX_EDU_NB,48.743
3,AFG,2020,SEX_T,EDU_AGGREGATE_BAS,LOW,UNE_TUNE_SEX_EDU_NB,124.640
4,AFG,2020,SEX_T,EDU_AGGREGATE_INT,MEDIUM,UNE_TUNE_SEX_EDU_NB,96.029
5,AFG,2020,SEX_T,EDU_AGGREGATE_ADV,HIGH,UNE_TUNE_SEX_EDU_NB,65.025


## 6. Vérifications de cohérence

In [12]:
print('=== VÉRIFICATIONS ===')
print()

# Doublons
dup_emp = emploi.duplicated(['country', 'year', 'education_level']).sum()
dup_une = chomage.duplicated(['country', 'year', 'education_level']).sum()
print(f'Doublons emploi  : {dup_emp}  (attendu : 0)')
print(f'Doublons chômage : {dup_une}  (attendu : 0)')
print()

# Valeurs nulles
print('Valeurs nulles emploi :')
print(emploi.isnull().sum())
print()
print('Valeurs nulles chômage :')
print(chomage.isnull().sum())
print()

# Pays en commun
pays_emp = set(emploi['country'].unique())
pays_une = set(chomage['country'].unique())
print(f'Pays en commun          : {len(pays_emp & pays_une)}')
print(f'Pays seulement emploi   : {len(pays_emp - pays_une)}')
print(f'Pays seulement chômage  : {len(pays_une - pays_emp)}')
print()

# Distribution des niveaux
print('Distribution education_level (emploi) :')
print(emploi['education_level'].value_counts())

=== VÉRIFICATIONS ===

Doublons emploi  : 0  (attendu : 0)
Doublons chômage : 0  (attendu : 0)

Valeurs nulles emploi :
country            0
year               0
sex                0
education_code     0
education_level    0
indicator          0
nb_employes        0
dtype: int64

Valeurs nulles chômage :
country            0
year               0
sex                0
education_code     0
education_level    0
indicator          0
nb_chomeurs        0
dtype: int64

Pays en commun          : 175
Pays seulement emploi   : 3
Pays seulement chômage  : 0

Distribution education_level (emploi) :
education_level
LOW       2045
MEDIUM    2034
HIGH      2029
Name: count, dtype: int64


## 7. Export des fichiers propres

In [14]:
# Export au format CSV dans data/clean/
emploi.to_csv(CLEAN_DIR / 'emploi_clean.csv', index=False, encoding='utf-8')
chomage.to_csv(CLEAN_DIR / 'chomage_clean.csv', index=False, encoding='utf-8')

print('Fichiers exportés avec succès :')
print('  data/clean/emploi_clean.csv')
print('  data/clean/chomage_clean.csv')

Fichiers exportés avec succès :
  data/clean/emploi_clean.csv
  data/clean/chomage_clean.csv
